In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Step 1: Import Earth Engine and other required libraries ---
import ee
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import datetime
import os
import seaborn as sns

# Global matplotlib settings
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='ee-abshirodkar15')

print("Earth Engine initialized successfully using project: ee-abshirodkar15")

In [ ]:
# Study area polygon (Pathum Thani)
Pathum_Thani = ee.Geometry.Polygon([
  [[100.49344515247942, 13.911774187300836],
   [100.78595613880755, 13.911774187300836],
   [100.78595613880755, 14.135610659931288],
   [100.49344515247942, 14.135610659931288],
   [100.49344515247942, 13.911774187300836]]
])

# Selected dates
selectedDateList = [
    '2024-06-21', '2024-06-29', '2024-07-07', '2024-07-15',
    '2024-07-23', '2024-07-31', '2024-08-08', '2024-08-16',
    '2024-08-24', '2024-09-01', '2024-09-09', '2024-09-17',
    '2024-10-19', '2024-10-27', '2024-11-04', '2024-11-12',
    '2024-11-20', '2024-11-28', '2024-12-06', '2024-12-22',
    '2024-12-30', '2025-01-07'
]

# Combined sampling points for all zones
samplingPoints = ee.FeatureCollection([
    # Zone 1
    ee.Feature(ee.Geometry.Point([100.620189, 14.082205]), {
        'Zone': 1, 'Station': 'Talad Thai Pt1'
    }),
    ee.Feature(ee.Geometry.Point([100.620189, 14.081055]), {
        'Zone': 1, 'Station': 'Talad Thai Pt2'
    }),
    ee.Feature(ee.Geometry.Point([100.620228, 14.077457]), {
        'Zone': 1, 'Station': 'Talad Thai Pt3'
    }),
    # Zone 2
    ee.Feature(ee.Geometry.Point([100.560725, 14.038922]), {
        'Zone': 2, 'Station': 'Khlong Bang Luang Pt1'
    }),
    ee.Feature(ee.Geometry.Point([100.554583, 14.040492]), {
        'Zone': 2, 'Station': 'Khlong Bang Luang Pt2'
    }),
    ee.Feature(ee.Geometry.Point([100.557736, 14.039386]), {
        'Zone': 2, 'Station': 'Khlong Bang Luang Pt3'
    }),
    ee.Feature(ee.Geometry.Point([100.557125, 14.040869]), {
        'Zone': 2, 'Station': 'Khlong Bang Luang Pt4'
    }),
    # Zone 3
    ee.Feature(ee.Geometry.Point([100.5585, 14.0534]), {
        'Zone': 3, 'Station': 'Ban Phrao Pt1'
    }),
    ee.Feature(ee.Geometry.Point([100.5557, 14.05214]), {
        'Zone': 3, 'Station': 'Ban Phrao Pt2'
    }),
    ee.Feature(ee.Geometry.Point([100.5553, 14.05188]), {
        'Zone': 3, 'Station': 'Ban Phrao Pt3'
    }),
    ee.Feature(ee.Geometry.Point([100.5542, 14.05183]), {
        'Zone': 3, 'Station': 'Ban Phrao Pt4'
    })
])

# TODO: Replace with actual asset name from thesis
phCombined = ee.FeatureCollection("projects/ee-st124278/assets/WQ_Combined/pH_Combined")

print(f"Study area and {samplingPoints.size().getInfo()} sampling points defined")

In [ ]:
# TODO: Replace with actual indices and asset name from thesis
def calculateIndices(image):
    return ee.Image(image).select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B7']).addBands([
        ee.Image(image).select('SR_B2').divide(ee.Image(image).select('SR_B4')).rename('B2_B4'),
        ee.Image(image).select('SR_B2').divide(ee.Image(image).select('SR_B3')).rename('B2_B3'),
        ee.Image(image).select('SR_B2').add(ee.Image(image).select('SR_B7')).rename('B2_B7'),
        ee.Image(image).select('SR_B1').rename('C')
    ])

# Function to get filtered collection for specified dates
def getFilteredCollection(collection, dates):
    return ee.ImageCollection.fromImages(
        ee.List(dates).map(lambda date:
            collection
            .filterDate(ee.Date(date).advance(-1, 'day'), ee.Date(date).advance(1, 'day'))
            .mean()
            .set('system:time_start', ee.Date(date).millis())
            .set('date', ee.Date(date))
        )
    )

# Function to prepare training data by matching satellite imagery with in-situ measurements
def prepareTrainingData(landsatImage, phFeatureCollection):
    image = ee.Image(landsatImage)

    # Calculate indices
    imageWithIndices = calculateIndices(image)

    # Get date
    imageDate = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')

    # Filter in-situ data for the current date
    dateFilteredpH = phFeatureCollection.filter(ee.Filter.equals('date', imageDate))

    # Extract features at sampling points
    trainingPoints = samplingPoints.map(lambda point:
        ee.Feature(point.geometry(), {
            'B2_B4': imageWithIndices.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point.geometry(),
                scale=30,
                maxPixels=1e9
            ).get('B2_B4'),
            'B2_B3': imageWithIndices.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point.geometry(),
                scale=30,
                maxPixels=1e9
            ).get('B2_B3'),
            'B2_B7': imageWithIndices.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point.geometry(),
                scale=30,
                maxPixels=1e9
            ).get('B2_B7'),
            'C': imageWithIndices.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point.geometry(),
                scale=30,
                maxPixels=1e9
            ).get('C'),
            'pH': ee.Number(ee.Algorithms.If(
                dateFilteredpH
                .filter(ee.Filter.equals('station', point.get('Station')))
                .filter(ee.Filter.equals('zone', point.get('Zone')))
                .first(),
                dateFilteredpH
                .filter(ee.Filter.equals('station', point.get('Station')))
                .filter(ee.Filter.equals('zone', point.get('Zone')))
                .first().get('pH'),
                None
            )),
            'Station': point.get('Station'),
            'Zone': point.get('Zone'),
            'date': imageDate
        })
    )

    return trainingPoints.filter(ee.Filter.notNull(['pH']))

print("Helper functions defined for data processing")

In [ ]:
# Define Landsat collection with correct scaling
landsatCollection = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
    .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')) \
    .filterBounds(Pathum_Thani) \
    .select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B7']) \
    .map(lambda image: image \
        .multiply(0.0000275) \
        .add(-0.2) \
        .copyProperties(image, ['system:time_start'])
    )

# Get filtered collection for selected dates
filteredCollection = getFilteredCollection(landsatCollection, selectedDateList)

print(f"Landsat collection prepared with {filteredCollection.size().getInfo()} images")

# Extract features for all dates
def collectAllFeatures():
    def processImage(image):
        imageDate = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')

        dateFilteredpH = phCombined.filter(ee.Filter.equals('date', imageDate))

        matchedPoints = samplingPoints \
            .filter(ee.Filter.inList('Station', dateFilteredpH.aggregate_array('station'))) \
            .filter(ee.Filter.inList('Zone', dateFilteredpH.aggregate_array('zone')))

        return prepareTrainingData(image, dateFilteredpH) \
            .filter(ee.Filter.inList('Station', matchedPoints.aggregate_array('Station'))) \
            .filter(ee.Filter.inList('Zone', matchedPoints.aggregate_array('Zone')))

    return filteredCollection.map(processImage).flatten()

allFeatures = collectAllFeatures()
allFeatures = allFeatures.filter(ee.Filter.notNull(['pH']))

print(f"Total features extracted: {allFeatures.size().getInfo()}")

In [ ]:
# Split dataset for training, validation, and testing
withRandom = allFeatures.randomColumn('random')
training = withRandom.filter(ee.Filter.lt('random', 0.65))
validation = withRandom.filter(ee.Filter.And(
    ee.Filter.gte('random', 0.65),
    ee.Filter.lt('random', 0.86)
))
testing = withRandom.filter(ee.Filter.gte('random', 0.86))

print('Total features:', allFeatures.size().getInfo())
print('Training set size:', training.size().getInfo())
print('Validation set size:', validation.size().getInfo())
print('Testing set size:', testing.size().getInfo())

# Feature standardization using training-set z-score statistics
def standardizeFeatures(trainingSet, validationSet, testingSet):
    features = ['B2_B4', 'B2_B3', 'B2_B7', 'C']
    means = {}
    stdDevs = {}

    for feat in features:
        meanVal = ee.Number(trainingSet.reduceColumns(
            reducer=ee.Reducer.mean(),
            selectors=[feat]
        ).get('mean'))

        stdVal = ee.Number(trainingSet.reduceColumns(
            reducer=ee.Reducer.stdDev(),
            selectors=[feat]
        ).get('stdDev'))

        means[feat] = meanVal
        stdDevs[feat] = stdVal

    meanDictEE = ee.Dictionary(means)
    stdDictEE = ee.Dictionary(stdDevs)

    def normalizeCollection(featureCollection):
        def normalize_feature(feature):
            normalizedProperties = {}

            for feat in features:
                val = ee.Number(feature.get(feat))
                mean = ee.Number(meanDictEE.get(feat))
                stdDev = ee.Number(stdDictEE.get(feat))

                norm = ee.Algorithms.If(
                    stdDev.gt(0),
                    val.subtract(mean).divide(stdDev),
                    val.subtract(mean)
                )

                normalizedProperties[feat] = norm

            return feature.set(normalizedProperties)

        return featureCollection.map(normalize_feature)

    normalizedTraining = normalizeCollection(trainingSet)
    normalizedValidation = normalizeCollection(validationSet)
    normalizedTesting = normalizeCollection(testingSet)

    return {
        'normalizedTraining': normalizedTraining,
        'normalizedValidation': normalizedValidation,
        'normalizedTesting': normalizedTesting,
        'means': meanDictEE,
        'stdDevs': stdDictEE
    }

print("Data split and normalization function defined")

In [ ]:
# Set seeds for reproducibility
import random
import numpy as np
random.seed(42)
np.random.seed(42)

# Main model execution function with Random Forest
def executeModel():
    print('Number of Landsat images:', filteredCollection.size().getInfo())

    normalizationResult = standardizeFeatures(training, validation, testing)
    normalizedTraining = normalizationResult['normalizedTraining']
    normalizedValidation = normalizationResult['normalizedValidation']
    normalizedTesting = normalizationResult['normalizedTesting']

    trainedModel = ee.Classifier.smileRandomForest(100, seed=42)
    trainedModel = trainedModel.setOutputMode('REGRESSION')
    trainedModel = trainedModel.train(
        features=normalizedTraining,
        classProperty='pH',
        inputProperties=['B2_B4', 'B2_B3', 'B2_B7', 'C']
    )

    trainingPredictions = normalizedTraining.classify(trainedModel)
    validationPredictions = normalizedValidation.classify(trainedModel)
    testingPredictions = normalizedTesting.classify(trainedModel)

    return {
        'model': trainedModel,
        'trainingPredictions': trainingPredictions,
        'validationPredictions': validationPredictions,
        'testingPredictions': testingPredictions,
        'means': normalizationResult['means'],
        'stdDevs': normalizationResult['stdDevs']
    }

In [ ]:
# Function to calculate performance metrics
def calculatePerformanceMetrics(predictions):
    def add_error_metrics(feature):
        predicted = ee.Number(feature.get('classification'))
        observed = ee.Number(feature.get('pH'))
        error = predicted.subtract(observed)
        sqError = error.pow(2)
        absError = error.abs()
        percentageError = error.divide(observed).multiply(100).abs()
        squaredPercentageError = percentageError.pow(2)

        return feature.set({
            'squared_error': sqError,
            'absolute_error': absError,
            'percentage_error': percentageError,
            'squared_percentage_error': squaredPercentageError
        })

    errorFeatures = predictions.map(add_error_metrics)

    meanAbsError = errorFeatures.aggregate_mean('absolute_error')
    meanSqError = errorFeatures.aggregate_mean('squared_error')
    meanAbsPercentageError = errorFeatures.aggregate_mean('percentage_error')
    meanSqPercentageError = errorFeatures.aggregate_mean('squared_percentage_error')

    rmse = ee.Number(meanSqError).sqrt()
    rmspe = ee.Number(meanSqPercentageError).sqrt()

    stats = errorFeatures.reduceColumns(
        reducer=ee.Reducer.pearsonsCorrelation(),
        selectors=['classification', 'pH']
    )

    r2 = ee.Number(stats.get('correlation')).pow(2)

    return ee.Dictionary({
        'n_samples': predictions.size(),
        'MAE': meanAbsError,
        'RMSE': rmse,
        'MAPE': meanAbsPercentageError,
        'RMSPE': rmspe,
        'R_squared': r2
    })

In [ ]:
# Function to create and save the validation plot for pH
def createImprovedValidationPlotpH(predictions, dataset_type, save_dir, start_val, end_val, unit_step):
    metrics = calculatePerformanceMetrics(predictions).getInfo()
    r_squared = metrics['R_squared']
    mape = metrics['MAPE']
    rmspe = metrics['RMSPE']

    data = predictions.select(['pH', 'classification']).getInfo()
    features = data['features']
    observed = [feature['properties']['pH'] for feature in features]
    predicted = [feature['properties']['classification'] for feature in features]

    df = pd.DataFrame({'Observed pH': observed, 'Predicted pH': predicted})

    plt.figure(figsize=(8, 6))

    sns.regplot(x='Observed pH', y='Predicted pH', data=df,
                scatter_kws={'alpha': 0.65, 'color': 'red', 's': 44, 'edgecolors': 'darkred', 'linewidths': 0.5},
                line_kws={'color': 'darkblue', 'linewidth': 2},
                ci=95, color='#8a89ff')

    ax = plt.gca()
    for poly in ax.collections:
        if type(poly).__name__ == 'PolyCollection':
            poly.set_alpha(0.25)

    plt.plot([start_val, end_val], [start_val, end_val], linestyle=':', color='black', linewidth=1.5)

    textstr = '\n'.join((
        f'RF $R^2$ = {r_squared:.2f}',
        f'MAPE = {mape:.2f}%',
        f'RMSPE = {rmspe:.2f}%'))

    props = dict(boxstyle='round,pad=0.5', facecolor='wheat', alpha=0.5, edgecolor='black', linewidth=1.0)
    plt.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=12,
             verticalalignment='top', bbox=props, color='black')

    plt.xlabel('Observed pH', fontsize=16.0, color='black')
    plt.ylabel('Predicted pH', fontsize=16.0, color='black')

    for spine in ['top', 'right', 'bottom', 'left']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color('black')
        ax.spines[spine].set_linewidth(1.0)

    ax.tick_params(axis='x', which='both', top=False, labelsize=12, labelcolor='black')
    ax.tick_params(axis='y', which='both', right=False, labelsize=12, labelcolor='black')
    plt.tick_params(axis='both', which='major', direction='out', length=4, width=1.0)

    plt.legend(['Data Points', 'Fitting Line', '95% Confidence Interval', '1:1 Line'],
               loc='lower right', fontsize=11, frameon=True, edgecolor='black', facecolor='white', framealpha=1.0)

    plt.xlim(start_val, end_val)
    plt.ylim(start_val, end_val)
    ax.set_aspect('equal', adjustable='box')

    plt.xticks(np.arange(start_val, end_val + unit_step, unit_step))
    plt.yticks(np.arange(start_val, end_val + unit_step, unit_step))

    filename = f"RF_{dataset_type.capitalize()}_Set_Plot_pH_1_1.png"
    filepath = os.path.join(save_dir, filename)
    plt.savefig(filepath, dpi=1200, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Plot for {dataset_type} set saved as {filepath}")

In [ ]:
# Execute model and compute performance metrics
results = None
save_directory = '/content/drive/MyDrive/WEM Thesis/pH/'

try:
    results = executeModel()

    trainingMetrics = calculatePerformanceMetrics(results['trainingPredictions'])
    validationMetrics = calculatePerformanceMetrics(results['validationPredictions'])
    testingMetrics = calculatePerformanceMetrics(results['testingPredictions'])

    print('pH Training Metrics:', trainingMetrics.getInfo())
    print('pH Validation Metrics:', validationMetrics.getInfo())
    print('pH Testing Metrics:', testingMetrics.getInfo())

    print('Model execution completed successfully')

except Exception as e:
    print('Error in workflow execution:', str(e))

In [ ]:
# Create validation plots for all three splits
if results is not None:
    try:
        start_val_ph = 6.5
        end_val_ph = 8.5
        unit_step_ph = 0.5

        print('Creating training set validation plot for pH')
        createImprovedValidationPlotpH(
            results['trainingPredictions'],
            'training',
            save_directory,
            start_val_ph,
            end_val_ph,
            unit_step_ph
        )

        print('Creating validation set validation plot for pH')
        createImprovedValidationPlotpH(
            results['validationPredictions'],
            'validation',
            save_directory,
            start_val_ph,
            end_val_ph,
            unit_step_ph
        )

        print('Creating testing set validation plot for pH')
        createImprovedValidationPlotpH(
            results['testingPredictions'],
            'testing',
            save_directory,
            start_val_ph,
            end_val_ph,
            unit_step_ph
        )
    except Exception as e:
        print("Error creating validation plots for pH:", str(e))
else:
    print("Skipping validation plots as model execution failed")